In [40]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as pgo

In [4]:
day_m1_data = pd.read_csv('round-2-data/prices_day_-1.csv',sep=";")
series_by_product = {}
products = day_m1_data['product'].drop_duplicates()


In [ ]:
for product in products:
    series_by_product[product] = day_m1_data[day_m1_data['product'] == product]

In [8]:
fig = px.line(series_by_product["CROISSANTS"],x="timestamp",y="ask_price_1")
fig.show()

In [38]:
# Start with CROISSANTS dataframe
croissants_df = series_by_product["CROISSANTS"].copy()
croissants_df.rename(columns={c: 'croissants_'+c for c in croissants_df.columns if c not in ['timestamp', 'product', 'day', 'profit_and_loss']}, inplace=True)

jams_df = series_by_product["JAMS"].copy()
jams_df.rename(columns={c: 'jams_'+c for c in jams_df.columns if c not in ['timestamp', 'product', 'day', 'profit_and_loss']}, inplace=True)
merged_df = pd.merge(croissants_df, jams_df, on="timestamp", how="inner")

djembe_df = series_by_product["DJEMBES"].copy()
djembe_df.rename(columns={c: 'djembe_'+c for c in djembe_df.columns if c not in ['timestamp', 'product', 'day', 'profit_and_loss']}, inplace=True)
merged_df = pd.merge(merged_df, djembe_df, on="timestamp", how="inner")



# Calculate expected prices for PICNIC_BASKET_1
prices = ['mid_price', 'ask_price_1', 'bid_price_1','ask_price_2', 'bid_price_2', 'ask_price_3', 'bid_price_3']
for thing in prices:
    merged_df[f'{thing}_b1'] = 6*merged_df[f'croissants_{thing}'] + 3*merged_df[f'jams_{thing}'] + 1*merged_df[f'djembe_{thing}']
    merged_df[f'{thing}_b2'] = 4*merged_df[f'croissants_{thing}'] + 2*merged_df[f'jams_{thing}']

# Store the result
series_by_product["PICNIC_BASKET_1_EXPECTED"] = merged_df[(f'{thing}_b1' for thing in prices)]

In [ ]:
## This figure shows the difference between the sum of the costs of basket 1 and the price of basket 1

fig = pgo.Figure()

fig.add_trace(pgo.Scatter(x=merged_df['timestamp'], y=merged_df['mid_price_b1'], mode='lines', name='expected_mid_price'))
fig.add_trace(pgo.Scatter(x=series_by_product["PICNIC_BASKET1"]['timestamp'], y=series_by_product["PICNIC_BASKET1"]['mid_price'], mode='lines', name='mid_price'))
fig.show()


In [ ]:
## This figure shows the difference between the sum of the costs of basket 2 and the price of basket 2

fig = pgo.Figure()

fig.add_trace(pgo.Scatter(x=merged_df['timestamp'], y=merged_df['mid_price_b2'], mode='lines', name='expected_mid_price'))
fig.add_trace(pgo.Scatter(x=series_by_product["PICNIC_BASKET2"]['timestamp'], y=series_by_product["PICNIC_BASKET2"]['mid_price'], mode='lines', name='mid_price'))
fig.show()
